This notebook defines the model that will be used to predict the performance of a new store for the swap engine.

The idea is to train a model that predicts:
- capture rate
- sales per sqm
- dwell time

Given inputs:
- gla of block
- gla category of block
- bl1_label
- avg & median window flow of store
- neighborhood synergy metric from synergy graph
- mall id
- mall total gla
- number of stores in mall
- mall category distribution (eg % of f&b, % of fashion, etc)

# Imports

In [ ]:
import pandas as pd

# Data Loading

In [ ]:
import constants.constants as cst
import constants.paths as pth

In [ ]:
# Dim Tables
dim_blocks = pd.read_csv(pth.INTERMEDIATE_DIM_BLOCKS, **cst.CSV_PARAMS)
dim_malls = pd.read_csv(pth.INTERMEDIATE_DIM_MALLS, **cst.CSV_PARAMS)

# Fact Tables
fact_stores = pd.read_csv(pth.INTERMEDIATE_FACT_STORES, **cst.CSV_PARAMS)
fact_malls = pd.read_csv(pth.INTERMEDIATE_FACT_MALLS, **cst.CSV_PARAMS)
fact_sri_scores = pd.read_csv(pth.INTERMEDIATE_FACT_SRI_SCORES, **cst.CSV_PARAMS)

# Store financials table
store_financials = pd.read_csv(pth.INTERMEDIATE_STORE_FINANCIALS, **cst.CSV_PARAMS)

# Cross visits table
cross_visits = pd.read_csv(pth.INTERMEDIATE_CROSS_VISITS, **cst.CSV_PARAMS)

# Data Preparation

In [ ]:
import constants.column_names as col

Compute beforehand:

In [ ]:
# For stores that span multiple blocks, we take the sum of the people window flow
# over all blocks
store_daily = (
    fact_stores.groupby([col.STORE_CODE, col.DATE])
    .agg(
        **{
            col.PEOPLE_WINDOW_FLOW: (col.PEOPLE_WINDOW_FLOW, "sum"),
        }
    )
    .reset_index()
)

store_daily.groupby(col.STORE_CODE).agg(
    **{
        col.MODEL_STORE_AVG_WINDOW_FLOW: (col.PEOPLE_WINDOW_FLOW, "mean"),
        col.MODEL_STORE_MEDIAN_WINDOW_FLOW: (col.PEOPLE_WINDOW_FLOW, "median"),
    }
)

# Feature Engineering

In [ ]:
def engineer_store_features(
    store_code: int,
    dim_blocks: pd.DataFrame,
    store_daily: pd.DataFrame,
    cross_visits: pd.DataFrame,
    affinity_matrix: pd.DataFrame,
) -> pd.DataFrame:
    """Engineer features for a given store.

    Args:
        store_code (int): The store code.
        dim_blocks (pd.DataFrame): Dimension table for blocks.
        store_daily (pd.DataFrame): Daily aggregated store data.
        cross_visits (pd.DataFrame): Cross visits data between stores.
        affinity_matrix (pd.DataFrame): Affinity matrix for categories.
        category_col (str, optional): Column name for category. Defaults to "bl1_label".

    Returns:
        dict: A dictionary of engineered features for the store.
    """
    features = {}

    # There are duplicate store codes in dim_blocks, correspond to stores that span over
    # multiple blocks. The gla corresponds to the sum of the gla of all blocks, so we
    # can take any of the rows for other store attributes.
    store_info = dim_blocks[dim_blocks[col.STORE_CODE] == store_code].iloc[0]
    mall_id = store_info[col.MALL_ID]
    store_category = store_info[col.CAT_HIGH]

    # Get neighboring stores based on cross visits
    store_cross = cross_visits[
        (cross_visits[col.STORE_CODE_1] == store_code)
        | (cross_visits[col.STORE_CODE_2] == store_code)
    ]

    neighbor_codes = set(store_cross[col.STORE_CODE_1]) | set(
        store_cross[col.STORE_CODE_2]
    )
    neighbor_codes.discard(store_code)

    # Get category distribution of neighboring stores
    neighbor_categories = dim_blocks[dim_blocks[col.STORE_CODE].isin(neighbor_codes)][
        col.CAT_HIGH
    ].value_counts()

    # Compute synergy score: sum of (affinity × neighbor_count) for each neighbor
    # category
    synergy_score = 0
    for neighbor_cat, count in neighbor_categories.items():
        if (
            store_category in affinity_matrix.index
            and neighbor_cat in affinity_matrix.columns
        ):
            affinity = affinity_matrix.loc[store_category, neighbor_cat]
            if pd.notna(affinity):
                synergy_score += affinity * count

    mall_stores = dim_blocks[dim_blocks[col.MALL_ID] == mall_id].drop_duplicates(
        subset=[col.STORE_CODE]
    )

    #### Features ####
    # Intrinsic store features
    features[col.MODEL_STORE_GLA] = store_info[col.GLA]
    features[col.MODEL_STORE_GLA_CAT] = store_info[col.GLA_CAT]
    features[col.MODEL_STORE_CATEGORY] = store_category

    # Location features
    features[col.MODEL_STORE_AVG_WINDOW_FLOW] = store_daily[
        store_daily[col.STORE_CODE] == store_code
    ][col.MODEL_STORE_AVG_WINDOW_FLOW]
    features[col.MODEL_STORE_MEDIAN_WINDOW_FLOW] = store_daily[
        store_daily[col.STORE_CODE] == store_code
    ][col.MODEL_STORE_MEDIAN_WINDOW_FLOW]

    # Neighborhood synergy feature
    features[col.MODEL_STORE_NEIGHBORHOOD_SYNERGY] = synergy_score
    features[col.MODEL_STORE_NB_NEIGHBORS] = len(neighbor_codes)

    # Mall features
    features[col.MODEL_MALL_TOTAL_GLA] = mall_stores[col.GLA].sum()
    features[col.MODEL_MALL_STORE_COUNT] = len(mall_stores)
    features[col.MODEL_MALL_CATEGORY_SHARE] = (
        mall_stores[mall_stores[col.CAT_HIGH] == store_category][col.GLA].sum()
        / features[col.MODEL_MALL_TOTAL_GLA]
    )

    return features

# Training Dataset Preparation